In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table
import utils.data_processing_silver_table
import utils.data_processing_gold_table


## set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 14:09:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [3]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [4]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
dates_str_lst

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01']

## Build Bronze Table

In [5]:
# run bronze backfill
for date_str in dates_str_lst:
    last_bronze_output = utils.data_processing_bronze_table.process_bronze_table_features(date_str, spark)

2023-01-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_01_01.csv
2023-01-01	row count: 530
saved to: datamart/bronze/bronze_features_attributes2023_01_01.csv
2023-01-01	row count: 530
saved to: datamart/bronze/bronze_features_financials2023_01_01.csv
2023-02-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_02_01.csv
2023-02-01	row count: 501
saved to: datamart/bronze/bronze_features_attributes2023_02_01.csv
2023-02-01	row count: 501
saved to: datamart/bronze/bronze_features_financials2023_02_01.csv
2023-03-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_03_01.csv
2023-03-01	row count: 506
saved to: datamart/bronze/bronze_features_attributes2023_03_01.csv
2023-03-01	row count: 506
saved to: datamart/bronze/bronze_features_financials2023_03_01.csv
2023-04-01	row count: 8974
saved to: datamart/bronze/bronze_feature_clickstream2023_04_01.csv
2023-04-01	row count: 510
saved to: datamart/bronze/bronze_feature

## Build Silver Table

In [7]:
import importlib
importlib.reload(utils.data_processing_silver_table)

# run silver backfill
for date_str in dates_str_lst:
    utils.data_processing_silver_table.process_silver_table_features(date_str, spark)

loaded from: datamart/bronze/bronze_feature_clickstream2023_01_01.csv row count: 8974
saved to: datamart/silver/silver_feature_clickstream2023_01_01.parquet
loaded from: datamart/bronze/bronze_features_attributes2023_01_01.csv row count: 530
saved to: datamart/silver/silver_features_attributes2023_01_01.parquet
loaded from: datamart/bronze/bronze_features_financials2023_01_01.csv row count: 530
saved to: datamart/silver/silver_features_financials2023_01_01.parquet
loaded from: datamart/bronze/bronze_feature_clickstream2023_02_01.csv row count: 8974
saved to: datamart/silver/silver_feature_clickstream2023_02_01.parquet
loaded from: datamart/bronze/bronze_features_attributes2023_02_01.csv row count: 501
saved to: datamart/silver/silver_features_attributes2023_02_01.parquet
loaded from: datamart/bronze/bronze_features_financials2023_02_01.csv row count: 501
saved to: datamart/silver/silver_features_financials2023_02_01.parquet
loaded from: datamart/bronze/bronze_feature_clickstream2023_03

In [8]:
import importlib
importlib.reload(utils.data_processing_gold_table)

gold_dfs = []

# run gold backfill
for date_str in dates_str_lst:
    df = utils.data_processing_gold_table.process_features_gold_table(date_str, spark)
    gold_dfs.append(df)

loaded from: datamart/silver/silver_feature_clickstream2023_01_01.parquet row count: 8974
loaded from: datamart/silver/silver_features_attributes2023_01_01.parquet row count: 530
loaded from: datamart/silver/silver_features_financials2023_01_01.parquet row count: 530
Final gold table row count after joins: 530
saved to: datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet
loaded from: datamart/silver/silver_feature_clickstream2023_02_01.parquet row count: 8974
loaded from: datamart/silver/silver_features_attributes2023_02_01.parquet row count: 501
loaded from: datamart/silver/silver_features_financials2023_02_01.parquet row count: 501
Final gold table row count after joins: 501
saved to: datamart/gold/feature_store/gold_feature_store_2023_02_01.parquet
loaded from: datamart/silver/silver_feature_clickstream2023_03_01.parquet row count: 8974
loaded from: datamart/silver/silver_features_attributes2023_03_01.parquet row count: 506
loaded from: datamart/silver/silver_features_

In [9]:
from functools import reduce

df_gold = reduce(lambda df1, df2: df1.union(df2), gold_dfs)

## EDA on credit labels

In [10]:
df_gold.describe().toPandas().T

,0,1,2,3,4
summary,count,mean,stddev,min,max
Customer_ID,8974,None,None,CUS_0x1000,CUS_0xffd
fe_1,8974,103.05794517494985,100.57982678811263,-307,460
fe_2,8974,103.22654334744819,99.54318620422787,-301,560
fe_3,8974,103.83374192110541,101.02931414280089,-256,465
fe_4,8974,105.33429908624916,100.57400239048744,-292,506
fe_5,8974,107.04357031424114,99.70407566450396,-258,504
fe_6,8974,100.3227100512592,99.2597629939472,-259,459
fe_7,8974,109.2198573657232,99.98712678837381,-240,508
fe_8,8974,110.14653443280588,101.02640432372485,-348,480


## Build gold table for labels

In [11]:
# create bronze datalake
gold_label_store_directory = "datamart/gold/label_store/"

if not os.path.exists(gold_label_store_directory):
    os.makedirs(gold_label_store_directory)

## inspect label store

In [12]:
folder_path = gold_label_store_directory
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

label_df.show()

row_count: 0
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-07-0

### Ignore for grading -- Model fitting and test

In [13]:
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

label_df_renamed = label_df.select(
    col("Customer_ID"),
    col("snapshot_date").alias("label_date"),
    col("label")
)

# Join features with labels
df_temp = df_gold.join(
    label_df_renamed,
    on="Customer_ID",
    how="inner"
).filter(
    col("snapshot_date") < col("label_date")
)

# Take the most recent features before each label date
window_spec = Window.partitionBy("Customer_ID", "label_date").orderBy(col("snapshot_date").desc())

df_model = df_temp.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1)

df_model = df_model.drop("rn", "label_date")

print(f"Rows matched: {df_model.count()}")
df_model.groupBy("label").count().show()

print(df.columns)

Rows matched: 8974


[Stage 1589:===================================================>  (23 + 1) / 24]

+-----+-----+
|label|count|
+-----+-----+
|    1| 2591|
|    0| 6383|
+-----+-----+

['Customer_ID', 'snapshot_date', 'fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Utilization_Ratio', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 'has_mortgage_loan', 'has_personal_loan', 'has_home_equity_loan', 'has_payday_loan', 'has_debt_consolidation_loan', 'has_student_loan', 'has_credit_builder_loan', 'has_not_specified', 'has_auto_loan', 'Credit_Mix_Label', 'Credit_History_Months', 'Spending_Level', 'Payment_Value']


In [14]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

# Check data
print(f"Total rows: {df_model.count()}")
print("\nLabel distribution:")
df_model.groupBy("label").count().show()

# Define feature columns (exclude identifiers and label)
exclude_cols = ["Customer_ID", "snapshot_date", "label", "loan_id", "label_def"]

# Combine exclusions
all_exclude = exclude_cols
feature_cols = [c for c in df_model.columns if c not in all_exclude]

# Separate categorical and numeric columns
categorical_cols = ['Occupation']  # Only Occupation remains as categorical
categorical_cols = [c for c in categorical_cols if c in df_model.columns]

binary_cols = ['has_mortgage_loan', 'has_personal_loan', 'has_home_equity_loan', 
               'has_payday_loan', 'has_debt_consolidation_loan', 'has_student_loan',
               'has_credit_builder_loan', 'has_not_specified', 'has_auto_loan']
binary_cols = [c for c in binary_cols if c in df_model.columns]

numeric_cols = [c for c in feature_cols if c not in categorical_cols + binary_cols]

print(f"Numeric: {len(numeric_cols)}, Categorical: {len(categorical_cols)}, Binary: {len(binary_cols)}")

# Handle null values
df_model = df_model.fillna(0, subset=numeric_cols)
df_model = df_model.fillna(0, subset=binary_cols)
df_model = df_model.fillna('Unknown', subset=categorical_cols)

# Index categorical columns
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") for c in categorical_cols]

# Assemble features
assembler_input_cols = numeric_cols + binary_cols + [f"{c}_index" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembler_input_cols, outputCol="features_raw")

# Scale features
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

# Random Forest
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, maxDepth=10, seed=42)

# Pipeline
pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])

# Split data
train_df, test_df = df_model.randomSplit([0.7, 0.3], seed=42)
print(f"\nTrain: {train_df.count()}, Test: {test_df.count()}")

# Train
print("\nTraining model...")
model = pipeline.fit(train_df)

# Predict
predictions = model.transform(test_df)

# Evaluate
auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction").evaluate(predictions)
accuracy = predictions.filter(col("label") == col("prediction")).count() / predictions.count()

print(f"\n=== RESULTS ===")
print(f"AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")

# Sample predictions
print("\nSample predictions:")
predictions.select("Customer_ID", "label", "prediction", "probability").show(10)

# Feature importance
importance = model.stages[-1].featureImportances
print("\nTop 10 features:")
for idx in sorted(range(len(importance)), key=lambda x: -importance[x])[:10]:
    print(f"  {assembler_input_cols[idx]}: {importance[idx]:.4f}")

Total rows: 8974

Label distribution:


+-----+-----+
|label|count|
+-----+-----+
|    1| 2591|
|    0| 6383|
+-----+-----+

Numeric: 37, Categorical: 1, Binary: 9



Train: 6373, Test: 2601

Training model...



=== RESULTS ===
AUC: 0.8009
Accuracy: 0.7974

Sample predictions:


[Stage 2441:======================================>               (17 + 7) / 24]

+-----------+-----+----------+--------------------+
|Customer_ID|label|prediction|         probability|
+-----------+-----+----------+--------------------+
| CUS_0x1011|    0|       0.0|[0.79312935792200...|
| CUS_0x104e|    0|       0.0|[0.86233698654713...|
| CUS_0x1063|    0|       0.0|[0.74609080821768...|
| CUS_0x1069|    0|       0.0|[0.90322276185723...|
| CUS_0x10eb|    0|       0.0|[0.79356445456137...|
| CUS_0x10ff|    0|       0.0|[0.82033664785530...|
| CUS_0x1100|    1|       1.0|[0.35750548822670...|
| CUS_0x1156|    0|       0.0|[0.52172477075681...|
| CUS_0x117d|    0|       0.0|[0.84094392724781...|
| CUS_0x119e|    0|       0.0|[0.76512165643591...|
+-----------+-----+----------+--------------------+
only showing top 10 rows


Top 10 features:
  Interest_Rate: 0.1301
  Num_Credit_Inquiries: 0.0690
  Credit_History_Months: 0.0608
  Num_Credit_Card: 0.0448
  Changed_Credit_Limit: 0.0405
  Num_of_Loan: 0.0344
  Num_Bank_Accounts: 0.0343
  fe_10: 0.0298
  fe_5: 0.0284
  f